In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [42]:
TARGET_COL = "load_load_mw"
PROJECT_ROOT = Path.cwd().resolve().parents[1]  # from src/modeling -> project root
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_PATH = DATA_DIR / "features.parquet"


def load_dataset(input_path: Path = FEATURES_PATH, target_col: str = TARGET_COL):
    """
    Load feature dataset and return X, y.

    Parameters:
    input_path (Path): Path to features parquet.
    target_col (str): Target column name.

    Returns:
    X (pd.DataFrame): Feature matrix.
    y (pd.Series): Target series.
    """
    if not input_path.exists():
        raise FileNotFoundError(f"Input file '{input_path}' not found.")

    df = pd.read_parquet(input_path)

    if df.empty:
        raise ValueError("The input dataframe is empty.")
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in the dataframe.")

    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if df.index.has_duplicates:
        raise ValueError("Datetime index contains duplicates.")

    df = df.sort_index()
    dt_index = pd.DatetimeIndex(df.index)
    inferred_freq = pd.infer_freq(dt_index)
    if inferred_freq is not None:
        df = df.asfreq(inferred_freq)

    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

In [43]:
X, y = load_dataset(FEATURES_PATH, TARGET_COL)
X.shape, y.shape

((37509, 150), (37509,))

In [44]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [45]:
def make_time_series_folds(
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = 5,
    test_size: int | None = None,
    gap: int = 0,
    verbose: bool = True,
):
    """Create reusable time-series CV folds for fair method comparison."""
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of rows")
    if not X.index.equals(y.index):
        raise ValueError("X and y must share the same index")
    if n_splits < 2:
        raise ValueError("n_splits must be >= 2")

    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap)
    folds = []

    for fold_id, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
        fold = {
            "fold": fold_id,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "X_train": X.iloc[train_idx],
            "y_train": y.iloc[train_idx],
            "X_val": X.iloc[val_idx],
            "y_val": y.iloc[val_idx],
        }
        folds.append(fold)

        if verbose:
            print(
                f"Fold {fold_id}: "
                f"train={len(train_idx)} ({X.index[train_idx[0]]} -> {X.index[train_idx[-1]]}) | "
                f"val={len(val_idx)} ({X.index[val_idx[0]]} -> {X.index[val_idx[-1]]})"
            )

    return folds

In [46]:
cv_folds = make_time_series_folds(
    X_h24,
    y_h24,
    n_splits=2,
    test_size=24 * 30*12,  # one month validation windows
    gap=0,             # avoid leakage around forecast boundary
    verbose=True,
 )

len(cv_folds)

Fold 1: train=20205 (2022-01-08 00:00:00+00:00 -> 2024-04-28 20:00:00+00:00) | val=8640 (2024-04-28 21:00:00+00:00 -> 2025-04-23 20:00:00+00:00)
Fold 2: train=28845 (2022-01-08 00:00:00+00:00 -> 2025-04-23 20:00:00+00:00) | val=8640 (2025-04-23 21:00:00+00:00 -> 2026-04-18 20:00:00+00:00)


2

In [39]:
def evaluate_naive_cv(cv_folds, y_current: pd.Series):
    """Evaluate day-ahead naive baseline on the same CV folds used by all methods."""
    rows = []

    for fold in cv_folds:
        fold_id = fold["fold"]
        y_val = fold["y_val"]
        val_idx = y_val.index

        # For h+24 setup, naive prediction at time t is current load y(t).
        y_pred = y_current.reindex(val_idx)

        fold_df = pd.DataFrame({"y_true": y_val, "y_pred": y_pred}).dropna()
        fold_mae = mean_absolute_error(fold_df["y_true"], fold_df["y_pred"])

        fold_rmse = np.sqrt(mean_squared_error(fold_df["y_true"], fold_df["y_pred"]))

        rows.append(
            {
                "fold": fold_id,
                "n": len(fold_df),
                "mae": fold_mae,
                "rmse": fold_rmse,
                "start": fold_df.index.min(),
                "end": fold_df.index.max(),
            }
        )

    results = pd.DataFrame(rows).set_index("fold")
    return results

naive_cv_results = evaluate_naive_cv(cv_folds, y)

naive_cv_results

,n,mae,rmse,start,end
fold,,,,,
1,8640,508.807707,739.660556,2024-04-28 21:00:00+00:00,2025-04-23 20:00:00+00:00
2,8640,449.995483,616.059549,2025-04-23 21:00:00+00:00,2026-04-18 20:00:00+00:00


In [40]:
def run_sarimax_easy(
    cv_folds,
    X_full: pd.DataFrame,
    y_full: pd.Series,
    tune: bool = True,
):
    """Simple SARIMAX-only pipeline with optional small CV tuning."""
    import warnings
    from statsmodels.tools.sm_exceptions import ConvergenceWarning

    # Small search space keeps runtime reasonable.
    candidates = [
        ((1, 0, 1), (1, 0, 1, 24)),
        ((2, 0, 1), (1, 0, 1, 24)),
        ((1, 0, 1), (1, 1, 1, 24)),
    ]
    if not tune:
        candidates = [candidates[0]]

    def prep_exog(X_df: pd.DataFrame, cols=None, medians=None):
        X_num = X_df.select_dtypes(include=["number"]).copy()
        if cols is None:
            cols = X_num.columns.tolist()
        X_num = X_num.reindex(columns=cols)
        if medians is None:
            medians = X_num.median(numeric_only=True)
        X_num = X_num.ffill().bfill().fillna(medians).fillna(0.0)
        return X_num.astype(float), cols, medians

    def eval_candidate(order, seasonal_order):
        rows = []
        for fold in cv_folds:
            y_train, y_val = fold["y_train"], fold["y_val"]
            X_train, X_val = fold["X_train"], fold["X_val"]

            X_train_exog, cols, medians = prep_exog(X_train)
            X_val_exog, _, _ = prep_exog(X_val, cols=cols, medians=medians)

            model = SARIMAX(
                endog=y_train.to_numpy(),
                exog=X_train_exog.to_numpy(),
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )

            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=ConvergenceWarning)
                fitted = model.fit(disp=False, method="lbfgs", maxiter=400)

            y_pred = pd.Series(
                fitted.forecast(steps=len(y_val), exog=X_val_exog.to_numpy()),
                index=y_val.index,
            )
            fold_df = pd.DataFrame({"y_true": y_val, "y_pred": y_pred}).dropna()
            rows.append(
                {
                    "fold": fold["fold"],
                    "mae": mean_absolute_error(fold_df["y_true"], fold_df["y_pred"]),
                    "rmse": np.sqrt(mean_squared_error(fold_df["y_true"], fold_df["y_pred"])),
                }
            )
        return pd.DataFrame(rows).set_index("fold")

    search_rows = []
    best_cv = None
    best_params = None
    best_rmse = np.inf

    for order, seasonal_order in candidates:
        try:
            cv_result = eval_candidate(order, seasonal_order)
            avg_rmse = cv_result["rmse"].mean()
            avg_mae = cv_result["mae"].mean()
            search_rows.append(
                {
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "avg_rmse": avg_rmse,
                    "avg_mae": avg_mae,
                }
            )
            if avg_rmse < best_rmse:
                best_rmse = avg_rmse
                best_params = (order, seasonal_order)
                best_cv = cv_result
        except Exception as exc:
            search_rows.append(
                {
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "avg_rmse": np.inf,
                    "avg_mae": np.inf,
                    "error": str(exc),
                }
            )

    search_results = pd.DataFrame(search_rows).sort_values("avg_rmse")

    best_order, best_seasonal_order = best_params
    X_full_exog, selected_columns, medians = prep_exog(X_full)
    final_model = SARIMAX(
        endog=y_full.to_numpy(),
        exog=X_full_exog.to_numpy(),
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, method="lbfgs", maxiter=400)

    artifacts = {
        "model": final_model,
        "order": best_order,
        "seasonal_order": best_seasonal_order,
        "selected_columns": selected_columns,
        "medians": medians,
    }

    return {
        "search_results": search_results,
        "cv_results": best_cv,
        "artifacts": artifacts,
    }

sarimax_output = run_sarimax_easy(
    cv_folds=cv_folds,
    X_full=X_h24,
    y_full=y_h24,
    tune=True,
 )

sarimax_search_results = sarimax_output["search_results"]
sarimax_cv_results = sarimax_output["cv_results"]
sarimax_artifacts = sarimax_output["artifacts"]

sarimax_search_results, sarimax_cv_results

KeyboardInterrupt: 